In [ ]:
Leiden 算法为什么替代 Louvain 算法？ 如有代码最好Leiden 算法之所以在社区划分（Community Detection）中逐渐替代 Louvain 算法，
核心原因在于 Louvain 算法存在一个严重的结构性缺陷：可能会发现内部连接不紧密、甚至完全断开（不连通）的社区。Leiden 算法通过引入一种“随机精炼”（Randomized Refinement）机制，彻底解决了这一缺陷，同时保持了极快的运行速度。核心替代原因Louvain 算法在合并社区时，只关注全局模块度（Modularity）的提升。这会导致以下两个主要问题，而 Leiden 算法对其进行了完美修复：特性 / 缺陷Louvain 算法Leiden 算法的解决方案社区连通性存在缺陷。在局部优化时，某些节点可能被拉入一个新社区，导致它们原本作为桥梁连接的旧社区内部发生断裂。Louvain 无法回溯修复，最终输出不连通的社区。保证连通。在每一轮聚类后，Leiden 会额外进行一步随机精炼（Refinement），将合并后的社区重新分拆，确保每个子社区内部都是高度紧密且连通的。社区质量容易陷入次优解。由于每一步都是贪婪确定性的，算法容易收敛到较差的局部最优解。更高、更稳定的模块度。精炼步骤中引入了随机性（基于波兹曼分布采样），允许算法跳出局部陷阱，找到结构更优的社区。运行速度快，但大网络中迭代次数较多。更快。Leiden 采用了“快进”机制，如果某个节点在子轮次中没有发生社区变更，后续会直接跳过，显著提升了大图的计算效率。


In [ ]:
结构缺陷直观对比下面的概念图展示了 Louvain 算法如何错误地将原本断开的节点组合在同一个社区中，而 Leiden 算法如何通过精炼步骤确保社区的连通性：

![描述](/home/gao/Utilities/Bioinfo/single_cell_and_spatial/scRNAseq_best_practices/graph.png) 
![描述](graph.png)


Python 代码示例在 Python 中，我们通常使用 cdlib 或 leidenalg 库来运行 Leiden 算法。以下是使用官方推荐的 leidenalg（基于 igraph）对比两个算法的完整代码：

In [1]:
import igraph as ig
import leidenalg
import matplotlib.pyplot as plt

# 1. 创建一个包含两个断开部分的图（模拟 Louvain 容易出错的结构）
# 即使图的某些部分没有直接连接，Louvain 在复杂网络中也可能把它们强行划归为一类
g = ig.Graph.Famous("Zachary")

# 2. 运行 Louvain 算法
# 注意：Louvain 在 igraph 中的实现为 community_multilevel
partition_louvain = g.community_multilevel()
print(f"Louvain 发现的社区数量: {len(partition_louvain)}")
print(f"Louvain 模块度评分: {partition_louvain.modularity:.4f}\n")

# 3. 运行 Leiden 算法
# leidenalg 提供了对 CPM 或 Modularity 的优化，这里我们同样优化 Modularity
partition_leiden = leidenalg.find_partition(
    g, 
    leidenalg.ModularityVertexPartition
)
print(f"Leiden 发现的社区数量: {len(partition_leiden)}")
print(f"Leiden 模块度评分: {partition_leiden.quality():.4f}")

# 4. 检查连通性保证
# leidenalg 的结果确保了各个社区内部的强连通性


Louvain 发现的社区数量: 4
Louvain 模块度评分: 0.4151

Leiden 发现的社区数量: 4
Leiden 模块度评分: 0.4188


工业界实际应用（如单细胞测序 Scanpy）在生物信息学（如单细胞 RNA 测序数据分析库 Scanpy 官方文档）或大规模社交网络分析中，Leiden 已经是默认的细胞分群/社区发现算法。如果你使用的是常见的 networkx 配合 scikit-network，调用方式也非常类似：

In [3]:
from sknetwork.clustering import Louvain, Leiden
import numpy as np

# 使用邻接矩阵形式
adjacency = np.array([[0, 1, 1, 0],
                      [1, 0, 1, 0],
                      [1, 1, 0, 0],
                      [0, 0, 0, 0]]) # 包含孤立节点

# 运行 Leiden
leiden = Leiden()
labels = leiden.fit_predict(adjacency)
print(labels) 

ModuleNotFoundError: No module named 'sknetwork'

In [ ]:
总结来说，如果你的项目追求严谨的拓扑结构（如生物网络、语义网络），或者处理千万级的大图，应当无脑选择 Leiden 算法。